::: {.sb-lesson-links}
[Exercises →](../exercises/titanic.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/06-ensemble-methods.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-workflow.ipynb){download="lesson-workflow.ipynb"}  
Data: [customers.csv](../downloads/data/customers.csv){download="customers.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

Let's train ensemble models on our customers dataset (predict whether a customer buys the product).

## What will you learn in this course? 🧐🧐
This lecture is a follow-along demo that will guide you through the steps of building ensemble models, choosing the base estimators, optimizing the hyperparameters and evaluating the performances on actual data.

* Preprocessing
    * Training pipeline
    * Test pipeline
* Bagging
    * Bagging with logistic regression as base estimator
    * Bagging with decision tree as base estimator
* Boosting
    * AdaBoost
    * scikit-learn's GradientBoosting
    * XGBoost
* Voting
* Stacking

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
# import ensemble methods
from sklearn.ensemble import BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from xgboost import XGBClassifier
# import base estimators
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, RocCurveDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"


In [ ]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("../data/customers.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [ ]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include='all')
display(data_desc)
print()

print("Percentage of missing values: ")
display(100*dataset.isnull().sum()/dataset.shape[0])


Number of rows : 400

Display of dataset: 


,Country,Age,Salary,Purchased
0,Germany,20.0,38500.0,No
1,Spain,33.0,32200.0,No
2,Italy,43.0,47900.0,No
3,Germany,62.0,54300.0,No
4,France,53.0,51800.0,Yes



Basics statistics: 


,Country,Age,Salary,Purchased
count,400,391.000000,400.000000,400
unique,4,NaN,NaN,2
top,France,NaN,NaN,No
freq,139,NaN,NaN,241
mean,NaN,42.437340,47013.500000,NaN
std,NaN,13.297408,14079.992301,NaN
min,NaN,20.000000,13000.000000,NaN
25%,NaN,31.000000,35800.000000,NaN
50%,NaN,43.000000,47250.000000,NaN
75%,NaN,54.000000,58150.000000,NaN



Percentage of missing values: 


Country      0.00
Age          2.25
Salary       0.00
Purchased    0.00
dtype: float64

## Preprocessing

In [ ]:
# Separate target variable Y from features X
print("Separating labels from features...")
target_variable = "Purchased"

X = dataset.drop(target_variable, axis = 1)
Y = dataset.loc[:,target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Separating labels from features...
...Done.

Y : 
0     No
1     No
2     No
3     No
4    Yes
Name: Purchased, dtype: str

X :
   Country   Age   Salary
0  Germany  20.0  38500.0
1    Spain  33.0  32200.0
2    Italy  43.0  47900.0
3  Germany  62.0  54300.0
4   France  53.0  51800.0


In [ ]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i,t in X.dtypes.items():
    if ('float' in str(t)) or ('int' in str(t)) :
        numeric_features.append(i)
    else :
        categorical_features.append(i)

print('Found numeric features ', numeric_features)
print('Found categorical features ', categorical_features)

Found numeric features  ['Age', 'Salary']
Found categorical features  ['Country']


In [ ]:
# Divide dataset Train set & Test set 
print("Dividing into train and test sets...")
# WARNING : don't forget stratify=Y for classification problems
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0, stratify = Y)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [ ]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])

In [ ]:
# Create pipeline for categorical features
categorical_transformer = OneHotEncoder(drop='first') # no missing values in categorical data, so we only need the OHE

In [ ]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()
# Label encoding
print("Encoding labels...")
print(Y_train.head())
encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
print("...Done")
print(Y_train[0:5])

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head()) 
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()
# Label encoding
print("Encoding labels...")
print(Y_test[0:5])
Y_test = encoder.transform(Y_test)
print("...Done")
print(Y_test[0:5])

Performing preprocessings on train set...
     Country   Age   Salary
76    France  44.0  44000.0
367  Germany  38.0  48900.0
255  Germany  65.0  64700.0
218    Italy  21.0  25200.0
195   France  48.0  55700.0
...Done.
[[ 0.10554507 -0.23953466  0.          0.          0.        ]
 [-0.34803305  0.10714279  1.          0.          0.        ]
 [ 1.69306849  1.2250007   1.          0.          0.        ]
 [-1.63317106 -1.56964408  0.          1.          0.        ]
 [ 0.40793048  0.5882462   0.          0.          0.        ]]

Encoding labels...
76      No
367     No
255    Yes
218     No
195    Yes
Name: Purchased, dtype: str
...Done
[0 0 1 0 1]
Performing preprocessings on test set...
     Country   Age   Salary
357  Germany  59.0  68000.0
228  Germany  46.0  61000.0
105  Germany  52.0  67300.0
115    Italy  41.0  49600.0
188  Germany  31.0  30200.0
...Done.
[[ 1.23949037  1.45847736  1.          0.          0.        ]
 [ 0.25673778  0.96322385  1.          0.          0.        

## Bagging
### Bagging with logistic regression as base estimator

In [ ]:
# Perform grid search
print("Grid search...")
logistic_regression = LogisticRegression(max_iter = 1000) # max_iter changed because of convergence warning
model = BaggingClassifier(logistic_regression)

# Grid of values to be tested
params = {
    'estimator__C': [0.01, 0.05, 0.1, 0.5,], # base_estimator__ prefix because C is a parameter from LogisticRegression! 
    'n_estimators': [5, 10, 20, 30] # n_estimators is a hyperparameter of the ensemble method
}
print(params)
gridsearch = GridSearchCV(model, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__C': [0.01, 0.05, 0.1, 0.5], 'n_estimators': [5, 10, 20, 30]}


...Done.
Best hyperparameters :  {'estimator__C': 0.05, 'n_estimators': 10}
Best validation accuracy :  0.8313936401575265

Accuracy on training set :  0.83125
Accuracy on test set :  0.8125


### Bagging with decision tree as base estimator

In [ ]:
# Perform grid search
print("Grid search...")
decision_tree = DecisionTreeClassifier()
model = BaggingClassifier(decision_tree)

# Grid of values to be tested
params = {
    'estimator__max_depth': [1, 2, 3],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__min_samples_split': [2, 3, 4],
    'n_estimators': [2, 4, 6, 8, 10]
}
print(params)
gridsearch = GridSearchCV(model, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__max_depth': [1, 2, 3], 'estimator__min_samples_leaf': [1, 2, 3], 'estimator__min_samples_split': [2, 3, 4], 'n_estimators': [2, 4, 6, 8, 10]}


...Done.
Best hyperparameters :  {'estimator__max_depth': 1, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 4, 'n_estimators': 10}
Best validation accuracy :  0.8376535590430847

Accuracy on training set :  0.840625
Accuracy on test set :  0.775


## Boosting
### Adaboost
#### Adaboost with logistic regression as base estimator

In [ ]:
# Perform grid search
print("Grid search...")
logistic_regression = LogisticRegression(max_iter = 1000) # max_iter changed because of convergence warning
model = AdaBoostClassifier(logistic_regression)

# Grid of values to be tested
params = {
    'estimator__C': [0.01, 0.05, 0.1, 0.5,], # base_estimator__ prefix because C is a parameter from LogisticRegression! 
    'n_estimators': [5, 10, 20, 30] # n_estimators is a hyperparameter of the ensemble method
}
print(params)
gridsearch = GridSearchCV(model, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__C': [0.01, 0.05, 0.1, 0.5], 'n_estimators': [5, 10, 20, 30]}


...Done.
Best hyperparameters :  {'estimator__C': 0.01, 'n_estimators': 5}
Best validation accuracy :  0.8158467054605302

Accuracy on training set :  0.815625
Accuracy on test set :  0.8125


#### Adaboost with decision tree as base estimator

In [ ]:
# Perform grid search
print("Grid search...")
decision_tree = DecisionTreeClassifier()
model = AdaBoostClassifier(decision_tree)

# Grid of values to be tested
params = {
    'estimator__max_depth': [1, 2, 3],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__min_samples_split': [2, 3, 4],
    'n_estimators': [2, 4, 6, 8, 10]
}
print(params)
gridsearch = GridSearchCV(model, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__max_depth': [1, 2, 3], 'estimator__min_samples_leaf': [1, 2, 3], 'estimator__min_samples_split': [2, 3, 4], 'n_estimators': [2, 4, 6, 8, 10]}


...Done.
Best hyperparameters :  {'estimator__max_depth': 2, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'n_estimators': 8}
Best validation accuracy :  0.8125844942103098

Accuracy on training set :  0.8375
Accuracy on test set :  0.7875


### Scikit-learn's GradientBoosting
#### Boosting with logistic regression as base estimator: forbidden ⛔️

In [ ]:
logistic_regression = LogisticRegression(max_iter = 1000)
try:
    model = GradientBoostingClassifier(estimator = logistic_regression)
except TypeError as error:
    print("This fails:", error)

This fails: GradientBoostingClassifier.__init__() got an unexpected keyword argument 'estimator'. Did you mean 'n_estimators'?


#### Boosting with decision tree as base estimator
`GradientBoostingClassifier` has no `estimator` parameter: in scikit-learn, gradient boosting always uses decision trees, and their hyperparameters (`max_depth`, `min_samples_leaf`...) are set directly on the model.

In [ ]:
# Perform grid search
print("Grid search...")
model = GradientBoostingClassifier()

# Grid of values to be tested
params = {
    'max_depth': [1, 2, 3], # no base_estimator_ prefix because these are all arguments of GradientBoostingClassifier
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [2, 3, 4],
    'n_estimators': [2, 4, 6, 8, 10]
}
print(params)
gridsearch = GridSearchCV(model, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'max_depth': [1, 2, 3], 'min_samples_leaf': [1, 2, 3], 'min_samples_split': [2, 3, 4], 'n_estimators': [2, 4, 6, 8, 10]}


...Done.
Best hyperparameters :  {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 4}
Best validation accuracy :  0.8188150238053252

Accuracy on training set :  0.8375
Accuracy on test set :  0.7875


## XGBoost
XGBoost is a different library, but it provides a [scikit-learn API](https://xgboost.readthedocs.io/en/stable/python/python_api.html#module-xgboost.sklearn) that allows to train a model as if it had been built from a scikit-learn class.

In [ ]:
# Perform grid search
print("Grid search...")
xgboost = XGBClassifier()

# Grid of values to be tested
params = {
    'max_depth': [2, 4, 6], # exactly the same role as in scikit-learn
    'min_child_weight': [1, 2, 3], # effect is more or less similar to min_samples_leaf and min_samples_split
    'n_estimators': [2, 4, 6, 8,] # exactly the same role as in scikit-learn
}
print(params)
gridsearch = GridSearchCV(xgboost, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'max_depth': [2, 4, 6], 'min_child_weight': [1, 2, 3], 'n_estimators': [2, 4, 6, 8]}


...Done.
Best hyperparameters :  {'max_depth': 2, 'min_child_weight': 2, 'n_estimators': 4}
Best validation accuracy :  0.8188150238053252

Accuracy on training set :  0.840625
Accuracy on test set :  0.7625


## Voting
Contrary to bagging and boosting, the voting classifier is meant to mix different base estimators. Let's see an example with three base estimators:

* logistic regression
* decision tree
* SVM with rbf kernel

In [ ]:
# Logistic regression
# Perform grid search
print("Grid search...")
logreg = LogisticRegression()

# Grid of values to be tested
params = {
    'C': [0.1, 1.0, 10.0]
}
logreg_opt = GridSearchCV(logreg, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
logreg_opt.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", logreg_opt.best_params_)
print("Best validation accuracy : ", logreg_opt.best_score_)
print()
print("Accuracy on training set : ", logreg_opt.score(X_train, Y_train))
print("Accuracy on test set : ", logreg_opt.score(X_test, Y_test))

Grid search...
...Done.
Best hyperparameters :  {'C': 0.1}
Best validation accuracy :  0.8282783753600188

Accuracy on training set :  0.828125
Accuracy on test set :  0.8125


In [ ]:
# Decision tree
# Perform grid search
print("Grid search...")
dt = DecisionTreeClassifier()

# Grid of values to be tested
params = {
    'max_depth': [1, 2, 3], 
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [2, 3, 4]
}
dt_opt = GridSearchCV(dt, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
dt_opt.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", dt_opt.best_params_)
print("Best validation accuracy : ", dt_opt.best_score_)
print()
print("Accuracy on training set : ", dt_opt.score(X_train, Y_train))
print("Accuracy on test set : ", dt_opt.score(X_test, Y_test))

Grid search...


...Done.
Best hyperparameters :  {'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 2}
Best validation accuracy :  0.8125551049197672

Accuracy on training set :  0.840625
Accuracy on test set :  0.775


In [ ]:
# SVM
# Perform grid search
print("Grid search...")
svm = SVC(kernel = 'rbf', probability = True)

# Grid of values to be tested
params = {
    'C': [0.1, 1.0, 10.0],
    'gamma': [0.1, 1.0, 10.0]
}
svm_opt = GridSearchCV(svm, param_grid = params, cv = 3) # cv : the number of folds to be used for CV
svm_opt.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", svm_opt.best_params_)
print("Best validation accuracy : ", svm_opt.best_score_)
print()
print("Accuracy on training set : ", svm_opt.score(X_train, Y_train))
print("Accuracy on test set : ", svm_opt.score(X_test, Y_test))

Grid search...


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

...Done.
Best hyperparameters :  {'C': 1.0, 'gamma': 0.1}
Best validation accuracy :  0.8344795156644919

Accuracy on training set :  0.840625
Accuracy on test set :  0.8125


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

In [ ]:
# Voting
voting = VotingClassifier(estimators=[("logistic", logreg), ("tree", dt), ("svm", svm)], voting='soft') # soft: use probabilities for voting
voting.fit(X_train, Y_train)
print("Accuracy on training set : ", voting.score(X_train, Y_train))
print("Accuracy on test set : ", voting.score(X_test, Y_test))

Accuracy on training set :  0.9125
Accuracy on test set :  0.7625


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## Stacking
As voting, the stacking concerns different base estimators, but this time they will be used to estimate independent probabilities that will be plugged as input of a final estimator. The default final estimator is LogisticRegression, but it can be changed using the `final_estimator` parameter.

In the examples below, the models `logreg`, `tree` and `svm` have already been optimized through a grid search.

In [ ]:
# Default: LogisticRegression will be used as final estimator
print('Training stacking classifier...')
stacking = StackingClassifier(estimators = [("logistic", logreg), ("tree", dt), ("svm", svm)], cv = 3)
preds = stacking.fit_transform(X_train, Y_train)
predictions = pd.DataFrame(preds, columns=stacking.named_estimators_.keys())
print('...Done.')
display(predictions)
print("Accuracy on training set : ", stacking.score(X_train, Y_train))
print("Accuracy on test set : ", stacking.score(X_test, Y_test))

Training stacking classifier...
...Done.


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

,logistic,tree,svm
0,0.295819,0.0,0.130106
1,0.213799,0.0,0.133661
2,0.905654,1.0,0.893670
3,0.013329,0.0,0.135504
4,0.606867,1.0,0.549027
...,...,...,...
315,0.035066,0.0,0.138947
316,0.759599,1.0,0.814396
317,0.054661,0.0,0.140729
318,0.831891,1.0,0.914152


Accuracy on training set :  0.81875
Accuracy on test set :  0.8




::: {.callout-tip title="Check for correlations"}
As the predictions used in stacking are supposed to be independent, it's a good practice to check the correlation matrix of the outputs from the different estimators. If some predictions have a strong correlation, it's better to re-train the stacking model by dropping one of the estimators
:::



In [ ]:
# Check the predictions are independent
# Correlation matrix
corr_matrix = predictions.corr().round(2)
fig = px.imshow(corr_matrix, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Correlation matrix")
fig.show()

In [ ]:
# Re-train by dropping the tree estimator because it's quite correlated with the logistic regression
print('Training stacking classifier...')
stacking = StackingClassifier(estimators = [("logistic", logreg), ("svm", svm)], cv = 3)
preds = stacking.fit_transform(X_train, Y_train)
predictions = pd.DataFrame(preds, columns=stacking.named_estimators_.keys())
print('...Done.')
display(predictions)
print("Accuracy on training set : ", stacking.score(X_train, Y_train))
print("Accuracy on test set : ", stacking.score(X_test, Y_test))

Training stacking classifier...


...Done.


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

,logistic,svm
0,0.295819,0.126083
1,0.213799,0.129630
2,0.905654,0.899419
3,0.013329,0.131470
4,0.606867,0.552751
...,...,...
315,0.035066,0.134909
316,0.759599,0.821285
317,0.054661,0.136690
318,0.831891,0.919316


Accuracy on training set :  0.840625
Accuracy on test set :  0.8125


In [ ]:
# DecisionTree as final estimator
print('Training stacking classifier...')
stacking = StackingClassifier(estimators = [("logistic", logreg), ("svm", svm)], final_estimator = DecisionTreeClassifier(), cv = 3)
preds = stacking.fit_transform(X_train, Y_train)
predictions = pd.DataFrame(preds, columns=stacking.named_estimators_.keys())
print('...Done.')
display(predictions)
print("Accuracy on training set : ", stacking.score(X_train, Y_train))
print("Accuracy on test set : ", stacking.score(X_test, Y_test))

Training stacking classifier...
...Done.


/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/tmp/claude-227905993/-volatile-home-sb283337-website/1612eefd-9ad8-402e-be72-1a6f17c096ee/scratchpad/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. 

,logistic,svm
0,0.295819,0.124031
1,0.213799,0.127533
2,0.905654,0.898208
3,0.013329,0.129350
4,0.606867,0.548794
...,...,...
315,0.035066,0.132747
316,0.759599,0.819190
317,0.054661,0.134506
318,0.831891,0.918348


Accuracy on training set :  0.753125
Accuracy on test set :  0.7375


::: {.sb-next}
**[Your turn: Ensemble methods exercises →](../exercises/titanic.ipynb)**
:::